In [1]:
import numpy as np, pandas as pd, joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import LeaveOneOut, cross_val_predict

META_COLS = ("dataset", "task_id", "best_model")

def load(path):
    df = pd.read_csv(path)
    feats = [c for c in df if c not in META_COLS and not c.startswith("score_")]
    return df, feats

def make_meta_model():
    return make_pipeline(
        SimpleImputer(strategy="median"),   # e.g. mean_skew missing when no numeric cols
        RandomForestClassifier(n_estimators=300, min_samples_leaf=2,
                               class_weight="balanced_subsample", random_state=0),
    )

clf_df, clf_feats = load("../data/meta_clf.csv")
reg_df, reg_feats = load("../data/meta_reg.csv")

In [2]:
def quick_check(df, feats, name):
    X, y = df[feats], df["best_model"]
    proba = cross_val_predict(make_meta_model(), X, y, cv=LeaveOneOut(), method="predict_proba")
    classes = np.sort(y.unique())
    top1 = (classes[proba.argmax(1)] == y).mean()
    top3 = np.mean([yi in classes[np.argsort(-p)[:3]] for yi, p in zip(y, proba)])
    majority = y.value_counts(normalize=True).iloc[0]
    print(f"{name}: top-1={top1:.2f} | top-3={top3:.2f} | majority baseline={majority:.2f}")

quick_check(clf_df, clf_feats, "Classification")
quick_check(reg_df, reg_feats, "Regression")

Classification: top-1=0.33 | top-3=0.82 | majority baseline=0.29
Regression: top-1=0.45 | top-3=0.74 | majority baseline=0.42


In [3]:
def fit_and_save(df, feats, path):
    model = make_meta_model().fit(df[feats], df["best_model"])
    joblib.dump({"model": model, "features": feats,
                 "classes": list(model.classes_)}, path)
    rf = model[-1]
    return pd.Series(rf.feature_importances_, index=feats).sort_values(ascending=False)

print(fit_and_save(clf_df, clf_feats, "../models/meta_clf.joblib").head(8), "\n")
print(fit_and_save(reg_df, reg_feats, "../models/meta_reg.joblib").head(8))

lm_stump           0.121143
lm_1nn             0.111908
class_entropy      0.087811
mean_kurt          0.079811
log_n_samples      0.077254
imbalance_ratio    0.074281
n_samples          0.072909
n_features         0.063643
dtype: float64 

target_kurt      0.113415
lm_linear        0.110129
mean_kurt        0.100699
mean_skew        0.082133
target_skew      0.072881
lm_stump         0.069819
target_cv        0.069526
log_n_samples    0.066857
dtype: float64
